# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id`s for reproducibility and clarity. All field, record set, and column identifiers are taken directly from the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
md = dataset.metadata
# Print name and description accessed through object attributes
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets and their fields by `@id` as defined in the Croissant schema.

In [ ]:
# List available record sets by their @id
print("Available record sets and their field @id's:")
record_set_infos = []
for record_set in md.record_sets:
    # Name and @id
    print(f"- Record set name: {record_set.name}, @id: {record_set.id}")
    fields_ids = [field.id for field in record_set.fields]
    print(f"  Fields (@id): {fields_ids}\n")
    record_set_infos.append({'@id': record_set.id, 'name': record_set.name, 'fields': fields_ids})

# Example: Print 1st record from the first record set
if md.record_sets:
    first_rs = md.record_sets[0].id
    print(f"\nExample records from record set '@id': {first_rs}")
    for i, rec in enumerate(dataset.records(record_set=first_rs)):
        print(rec)
        if i > 1:
            break

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. All record set `@id`s are used directly.

In [ ]:
# Extract data from all record sets
record_sets = [rs.id for rs in md.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    if not df.empty:
        print(f"Columns (@id): {df.columns.tolist()}")
        display(df.head(2))  # Preview
    else:
        print("No data in this record set.")

# Choose a record set with substantial data for subsequent analysis
main_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rsid
        break
# For reproducibility, print the chosen record set @id
print(f"\nMain record set for analysis: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply typical EDA, such as filtering, normalization, and grouping. In this section, all fields are referenced by their Croissant `@id`.

We'll select a numeric field from the main record set for illustration.

In [ ]:
df = dataframes[main_record_set_id]
print("All columns in main record set:")
print(list(df.columns))

# Try some common field names for mass spec / protein data
potential_numeric_fields = [col for col in df.columns if any(s in col.lower() for s in ['abundance', 'mw', 'molecular_weight', 'count', 'peptide', 'score'])]
print('Potential numeric fields:', potential_numeric_fields)

# Try to pick one
numeric_field = None
for col in potential_numeric_fields:
    try:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if df[col].notna().sum() > 0:
            numeric_field = col
            break
    except Exception:
        continue
if numeric_field is None:
    numeric_field = df.select_dtypes(include='number').columns[0] if len(df.select_dtypes(include='number').columns) else df.columns[0]
print(f"Selected numeric field for analysis (@id): {numeric_field}")

# Filter > threshold (e.g., mean or example threshold)
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field (e.g., protein type or sample)
potential_group_fields = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
group_field = None
for col in potential_group_fields:
    unique_vals = df[col].nunique()
    if 1 < unique_vals < 50:  # Not too high cardinality
        group_field = col
        break
if group_field:
    print(f"Grouping by field (@id): {group_field}")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    display(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All axes use Croissant schema `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30)
plt.title(f'Distribution of {numeric_field} (@id)')
plt.xlabel(numeric_field)
plt.show()

# If grouping is possible, visualizing group means
if group_field:
    plt.figure(figsize=(10,4))
    order = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).index
    sns.barplot(x=group_field, y=numeric_field, data=filtered_df, order=order)
    plt.title(f'{numeric_field} mean per {group_field} (@id)')
    plt.xticks(rotation=90)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² mass spectrometry dataset using `mlcroissant`, referencing all entities using their schema `@id`s. We identified key fields for analysis, performed basic filtering and normalization, and visualized data distributions. You may extend this notebook with additional domain-specific analyses using the convenient programmatic interface provided by `mlcroissant`.